# Step 2: Audience Success Prediction

This notebook adds a machine learning extension on top of the completed data analysis module. The goal is not to replace the EDA conclusions, but to extend the project from **describing factors associated with movie success** to **predicting whether a movie is likely to be audience-successful**.

The prediction target is deliberately narrow and explainable:

`success_movie = 1` when `vote_average >= 7.0` and `vote_count >= 50`; otherwise `0`.

This is an **audience-response success** label. It is not a revenue, profit, or ROI label.

## 1. Setup and Scope

The model uses `outputs/cleaned_movies.csv` from the Step 1 cleaning pipeline. It keeps the same project direction and avoids recommendation-system or LLM-related features.

Important modeling choices:

- `vote_average` and `vote_count` define the label, so they are excluded from input features to avoid data leakage.
- Because successful movies are rare under this definition, accuracy is not the main metric. Precision, recall, F1-score, ROC-AUC, and especially PR-AUC are more informative.
- `popularity` is treated carefully because it may reflect post-release attention. This notebook compares models with and without `popularity`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import (
    FEATURE_SETS,
    MODEL_FEATURES,
    STRICT_MODEL_FEATURES,
    TARGET_COLUMN,
    build_models,
    create_success_dataset,
    cross_validate_model,
    evaluate_model,
    load_cleaned_movies,
    plot_confusion_matrix,
    plot_cv_metrics,
    plot_feature_importance,
    plot_model_comparison,
    plot_popularity_comparison,
    plot_threshold_tuning,
    random_forest_feature_importance,
    save_dataframe,
    save_metrics,
    save_models,
    split_features,
    threshold_tuning_table,
    train_and_evaluate_models,
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
CLEANED_PATH = OUTPUT_DIR / "cleaned_movies.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)

## 2. Load Cleaned Dataset

The ML module starts from the cleaned movie-level metadata created in Step 1. It does not merge `credits.csv`, `keywords.csv`, or ratings files in this stage.

In [2]:
movies = load_cleaned_movies(CLEANED_PATH)
print(f"Loaded cleaned dataset: {movies.shape[0]:,} rows, {movies.shape[1]:,} columns")
movies.head()

Loaded cleaned dataset: 45,433 rows, 32 columns


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,movie_id,release_year,genre_list,main_genre,profit,roi,budget_group,rating_group
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,862.0,1995.0,"['Animation', 'Comedy', 'Family']",Animation,343554033.0,11.451801,High,High
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,8844.0,1995.0,"['Adventure', 'Fantasy', 'Family']",Adventure,197797249.0,3.043035,High,High
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.712900,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0,15602.0,1995.0,"['Romance', 'Comedy']",Romance,NaN,NaN,NaN,Medium
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0,31357.0,1995.0,"['Comedy', 'Drama', 'Romance']",Comedy,65452156.0,4.090760,Medium,Medium
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0,11862.0,1995.0,['Comedy'],Comedy,NaN,NaN,NaN,Medium


## 3. Define Audience-Success Label

A movie is labeled as successful only when it has both a high rating and enough votes. This avoids treating a highly rated movie with only a few votes as equally reliable.

In [3]:
model_df, label_summary = create_success_dataset(
    movies,
    rating_threshold=7.0,
    vote_count_threshold=50,
)

label_summary_table = pd.DataFrame(
    [
        ("Initial cleaned rows", label_summary["initial_rows"]),
        ("Rows used for modeling", label_summary["model_rows"]),
        ("Successful movies", label_summary["positive_rows"]),
        ("Non-successful movies", label_summary["negative_rows"]),
        ("Positive class rate", f"{label_summary['positive_rate']:.2%}"),
        ("Rating threshold", label_summary["rating_threshold"]),
        ("Vote-count threshold", label_summary["vote_count_threshold"]),
    ],
    columns=["Item", "Value"],
)
label_summary_table

,Item,Value
0,Initial cleaned rows,45433
1,Rows used for modeling,45349
2,Successful movies,2502
3,Non-successful movies,42847
4,Positive class rate,5.52%
5,Rating threshold,7.0
6,Vote-count threshold,50


## 4. Class Imbalance Check

The positive class is rare. A naive model that always predicts `Not success` can get high accuracy, so accuracy alone would be misleading. This is why the notebook includes a Dummy baseline and focuses more on precision, recall, F1-score, and PR-AUC.

In [4]:
positive_rate = label_summary["positive_rate"]
most_frequent_accuracy = 1 - positive_rate
imbalance_table = pd.DataFrame(
    [
        ("Positive class rate", positive_rate),
        ("Naive most-frequent accuracy", most_frequent_accuracy),
        ("Why this matters", "A high accuracy can still miss every successful movie."),
    ],
    columns=["Metric", "Value"],
)
imbalance_table

,Metric,Value
0,Positive class rate,0.055172
1,Naive most-frequent accuracy,0.944828
2,Why this matters,A high accuracy can still miss every successfu...


## 5. Feature Engineering and Leakage Control

The model uses metadata-style features. It excludes `vote_average` and `vote_count` because those columns directly define the target.

Added features:

- `has_budget`: whether a positive budget was reported before invalid budget values were set to missing.
- `is_english`: whether `original_language` is English.
- `movie_age`: `2017 - release_year`, using 2017 as the dataset-era reference year.

For numeric missing values, the pipeline uses median imputation with missing indicators. This is especially important for `budget`, because many movies have invalid or missing budget data.

In [5]:
feature_table = pd.DataFrame(
    {
        "Feature set": ["Main model with popularity", "Strict model without popularity"],
        "Features": [", ".join(MODEL_FEATURES), ", ".join(STRICT_MODEL_FEATURES)],
    }
)
feature_table

,Feature set,Features
0,Main model with popularity,"budget, runtime, popularity, release_year, has..."
1,Strict model without popularity,"budget, runtime, release_year, has_budget, is_..."


In [6]:
missing_model_table = (
    pd.Series(label_summary["missing_after_model_cleaning"], name="missing_rows")
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values("missing_rows", ascending=False)
)
missing_model_table

,feature,missing_rows
0,budget,36469
1,runtime,1923
3,release_year,84
4,movie_age,84
2,popularity,0


## 6. Train/Test Split

A stratified split keeps the success/non-success ratio similar in the training and test sets.

In [7]:
X_train, X_test, y_train, y_test = split_features(
    model_df,
    features=MODEL_FEATURES,
    test_size=0.2,
    random_state=42,
)

split_table = pd.DataFrame(
    [
        ("Training rows", len(X_train), y_train.mean()),
        ("Test rows", len(X_test), y_test.mean()),
    ],
    columns=["Split", "Rows", "Positive rate"],
)
split_table

,Split,Rows,Positive rate
0,Training rows,36279,0.055183
1,Test rows,9070,0.055127


## 7. Model Training and Comparison

The comparison includes:

1. **Dummy Baseline**: always predicts the most frequent class.
2. **Logistic Regression**: an interpretable linear baseline with class weighting.
3. **Random Forest**: a non-linear tree model that can capture feature interactions.

The Dummy baseline is essential because the dataset is highly imbalanced.

In [8]:
model_names = ["Dummy Baseline", "Logistic Regression", "Random Forest"]
models, metrics_df, details = train_and_evaluate_models(
    X_train,
    X_test,
    y_train,
    y_test,
    features=MODEL_FEATURES,
    random_state=42,
    model_names=model_names,
)

metrics_path = save_metrics(metrics_df, OUTPUT_DIR / "ml_model_metrics.csv")
comparison_fig = plot_model_comparison(metrics_df, FIGURES_DIR / "ml_model_comparison.png")
metrics_df

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Random Forest,0.915546,0.366197,0.728,0.487282,0.947543,0.490864
1,Logistic Regression,0.851929,0.249852,0.842,0.385355,0.915835,0.370926
2,Dummy Baseline,0.944873,0.000000,0.000,0.000000,0.500000,0.055127


### Interpretation

The Dummy baseline may show high accuracy because most movies are not labeled as successful. However, it has zero recall and zero F1-score for the success class, meaning it does not identify any successful movies. A useful model should improve recall and F1-score, not only accuracy.

In [9]:
rf_row = metrics_df.loc[metrics_df["model"] == "Random Forest"].iloc[0]
baseline_row = metrics_df.loc[metrics_df["model"] == "Dummy Baseline"].iloc[0]

print(
    f"Dummy baseline: accuracy={baseline_row['accuracy']:.3f}, "
    f"recall={baseline_row['recall']:.3f}, F1={baseline_row['f1']:.3f}"
)
print(
    f"Random Forest: accuracy={rf_row['accuracy']:.3f}, precision={rf_row['precision']:.3f}, "
    f"recall={rf_row['recall']:.3f}, F1={rf_row['f1']:.3f}, "
    f"PR-AUC={rf_row['pr_auc']:.3f}"
)

Dummy baseline: accuracy=0.945, recall=0.000, F1=0.000
Random Forest: accuracy=0.916, precision=0.366, recall=0.728, F1=0.487, PR-AUC=0.491


## 8. Confusion Matrix

The confusion matrix shows the trade-off between identifying successful movies and avoiding false positives.

In [10]:
primary_model_name = "Random Forest"
confusion_fig = plot_confusion_matrix(
    details[primary_model_name]["confusion_matrix"],
    primary_model_name,
    FIGURES_DIR / "ml_confusion_matrix.png",
)
print(f"Saved confusion matrix to: {confusion_fig}")

Saved confusion matrix to: C:\Users\刘子豪\Desktop\code\IMDB_analysis\outputs\figures\ml_confusion_matrix.png


## 9. Popularity Feature Risk Check

`popularity` is informative, but it may represent post-release attention. Therefore, the model with `popularity` should not be described as a pure pre-release prediction model.

To make this limitation explicit, we compare Random Forest using:

- the main feature set with `popularity`
- a stricter feature set without `popularity`

In [11]:
popularity_rows = []
for feature_set_name, features in FEATURE_SETS.items():
    rf_model = build_models(features=features, random_state=42)["Random Forest"]
    rf_model.fit(X_train[features], y_train)
    metrics = evaluate_model(rf_model, X_test[features], y_test)
    popularity_rows.append(
        {
            "feature_set": feature_set_name,
            **{metric: metrics[metric] for metric in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]},
        }
    )

popularity_comparison_df = pd.DataFrame(popularity_rows)
save_dataframe(popularity_comparison_df, OUTPUT_DIR / "ml_popularity_comparison.csv")
plot_popularity_comparison(popularity_comparison_df, FIGURES_DIR / "ml_popularity_comparison.png")
popularity_comparison_df

,feature_set,accuracy,precision,recall,f1,roc_auc,pr_auc
0,With popularity,0.915546,0.366197,0.728,0.487282,0.947543,0.490864
1,Without popularity,0.890739,0.245067,0.472,0.322625,0.801522,0.253606


### Interpretation

If the model with `popularity` performs better, that does not automatically mean it is better for pre-release prediction. It may be using information related to audience attention after release. The stricter model without `popularity` gives a more conservative reference point.

## 10. Threshold Tuning

The default classification threshold is 0.5. In an imbalanced task, changing the threshold can shift the model between higher recall and higher precision.

- A lower threshold usually finds more successful movies but creates more false positives.
- A higher threshold usually reduces false positives but misses more successful movies.

In [12]:
threshold_df = threshold_tuning_table(
    models[primary_model_name],
    X_test[MODEL_FEATURES],
    y_test,
    thresholds=[0.3, 0.4, 0.5, 0.6, 0.7],
)
save_dataframe(threshold_df, OUTPUT_DIR / "ml_threshold_tuning.csv")
plot_threshold_tuning(threshold_df, FIGURES_DIR / "ml_threshold_tuning.png")
threshold_df

,threshold,precision,recall,f1,positive_predictions
0,0.3,0.248796,0.930,0.392571,1869
1,0.4,0.304755,0.846,0.448093,1388
2,0.5,0.366197,0.728,0.487282,994
3,0.6,0.445573,0.614,0.516400,689
4,0.7,0.539683,0.476,0.505845,441


## 11. Stratified 5-Fold Cross-Validation

A single train/test split can be affected by random variation. Stratified 5-fold cross-validation gives a more stable estimate while preserving the class ratio in each fold.

In [13]:
cv_model = build_models(features=MODEL_FEATURES, random_state=42)[primary_model_name]
cv_df = cross_validate_model(
    cv_model,
    model_df[MODEL_FEATURES],
    model_df[TARGET_COLUMN],
    cv_splits=5,
    random_state=42,
)
save_dataframe(cv_df, OUTPUT_DIR / "ml_cross_validation_metrics.csv")
plot_cv_metrics(cv_df, FIGURES_DIR / "ml_cv_metrics.png")
cv_df

,metric,mean,std,folds
0,precision,0.366136,0.005407,5
1,recall,0.760200,0.015194,5
2,f1,0.494161,0.005541,5
3,roc_auc,0.946350,0.001282,5
4,pr_auc,0.509296,0.010681,5


## 12. Feature Importance

Random Forest feature importance is a model-based indicator. It should be interpreted as association within the trained model, not as causal proof.

In [14]:
rf_importance = random_forest_feature_importance(models[primary_model_name], top_n=15)
save_dataframe(rf_importance, OUTPUT_DIR / "ml_feature_importance.csv")
importance_fig = plot_feature_importance(rf_importance, FIGURES_DIR / "ml_feature_importance.png")
rf_importance

,feature,importance
0,popularity,0.556655
1,runtime,0.100633
2,missingindicator_budget,0.048551
3,movie_age,0.046811
4,release_year,0.046457
5,has_budget,0.046051
6,budget,0.042760
7,main_genre_Unknown,0.009085
8,is_english,0.008056
9,main_genre_Horror,0.007987


## 13. Save Models and Written Summary

The trained model files and markdown summary are saved under `outputs/` for use in the Streamlit dashboard and presentation.

In [15]:
model_paths = save_models(models, MODELS_DIR)
model_paths

{'Dummy Baseline': WindowsPath('C:/Users/刘子豪/Desktop/code/IMDB_analysis/outputs/models/dummy_baseline.joblib'),
 'Logistic Regression': WindowsPath('C:/Users/刘子豪/Desktop/code/IMDB_analysis/outputs/models/logistic_regression.joblib'),
 'Random Forest': WindowsPath('C:/Users/刘子豪/Desktop/code/IMDB_analysis/outputs/models/random_forest.joblib')}

In [16]:
best_row = metrics_df.sort_values(["f1", "pr_auc"], ascending=False).iloc[0]
best_threshold = threshold_df.sort_values("f1", ascending=False).iloc[0]
cv_f1 = cv_df.loc[cv_df["metric"] == "f1"].iloc[0]
cv_pr_auc = cv_df.loc[cv_df["metric"] == "pr_auc"].iloc[0]
strict_row = popularity_comparison_df.loc[popularity_comparison_df["feature_set"] == "Without popularity"].iloc[0]
with_pop_row = popularity_comparison_df.loc[popularity_comparison_df["feature_set"] == "With popularity"].iloc[0]
top_features = rf_importance["feature"].head(5).tolist()

summary_lines = [
    "# IMDB Movie Success Prediction Summary",
    "",
    "## Project Objective",
    "This machine learning extension predicts whether a movie is audience-successful using the cleaned movie metadata from Step 1.",
    "The label represents audience response, not financial success.",
    "",
    "## Target Definition",
    f"A movie is labeled `success_movie = 1` when `vote_average >= {label_summary['rating_threshold']}` and `vote_count >= {label_summary['vote_count_threshold']}`.",
    "`vote_average` and `vote_count` are excluded from model features to prevent data leakage.",
    "",
    "## Class Imbalance",
    f"Rows used for modeling: {label_summary['model_rows']:,}.",
    f"Successful movies: {label_summary['positive_rows']:,} ({label_summary['positive_rate']:.2%}).",
    f"Non-successful movies: {label_summary['negative_rows']:,}.",
    f"The Dummy baseline reaches {baseline_row['accuracy']:.3f} accuracy but {baseline_row['recall']:.3f} recall and {baseline_row['f1']:.3f} F1-score, showing why accuracy is not the main metric.",
    "",
    "## Feature Engineering",
    "Features include budget, runtime, popularity, release year, language, main genre, has_budget, is_english, and movie_age.",
    "Numeric missing values are handled with median imputation plus missing indicators, which is important because budget has many invalid or missing values.",
    "",
    "## Model Comparison",
    f"The best model by F1-score is {best_row['model']} with accuracy {best_row['accuracy']:.3f}, precision {best_row['precision']:.3f}, recall {best_row['recall']:.3f}, F1-score {best_row['f1']:.3f}, ROC-AUC {best_row['roc_auc']:.3f}, and PR-AUC {best_row['pr_auc']:.3f}.",
    f"Random Forest achieves precision {rf_row['precision']:.3f}, recall {rf_row['recall']:.3f}, F1-score {rf_row['f1']:.3f}, and PR-AUC {rf_row['pr_auc']:.3f}.",
    "This means it can identify many audience-successful movies, but precision is still limited because successful movies are rare.",
    "",
    "## Popularity Feature Boundary",
    "The model with popularity uses platform attention information, so it should not be described as a pure pre-release prediction model.",
    f"With popularity: F1 {with_pop_row['f1']:.3f}, PR-AUC {with_pop_row['pr_auc']:.3f}.",
    f"Without popularity: F1 {strict_row['f1']:.3f}, PR-AUC {strict_row['pr_auc']:.3f}.",
    "The stricter model without popularity is useful as a conservative comparison.",
    "",
    "## Threshold Tuning",
    f"Among the tested thresholds, {best_threshold['threshold']:.1f} gives the highest F1-score ({best_threshold['f1']:.3f}).",
    "Lower thresholds favor recall, while higher thresholds favor precision.",
    "",
    "## Cross-Validation",
    f"Stratified 5-fold cross-validation for Random Forest gives mean F1 {cv_f1['mean']:.3f} +/- {cv_f1['std']:.3f} and mean PR-AUC {cv_pr_auc['mean']:.3f} +/- {cv_pr_auc['std']:.3f}.",
    "This reduces dependence on a single train/test split.",
    "",
    "## Feature Importance",
    f"Top Random Forest features: {', '.join(top_features)}.",
    "Feature importance should be read as model association, not causal proof.",
    "",
    "## Limitations",
    "The success label depends on chosen rating and vote-count thresholds.",
    "The dataset is highly imbalanced, so precision remains limited.",
    "Popularity may include post-release attention, so the model is exploratory rather than production-level or pure pre-release prediction.",
    "Budget is missing for many movies, and imputation cannot fully recover true budget information.",
    "",
    "## Generated Files",
    "- outputs/ml_model_metrics.csv",
    "- outputs/ml_popularity_comparison.csv",
    "- outputs/ml_threshold_tuning.csv",
    "- outputs/ml_cross_validation_metrics.csv",
    "- outputs/ml_feature_importance.csv",
    "- outputs/models/dummy_baseline.joblib",
    "- outputs/models/logistic_regression.joblib",
    "- outputs/models/random_forest.joblib",
    "- outputs/figures/ml_model_comparison.png",
    "- outputs/figures/ml_confusion_matrix.png",
    "- outputs/figures/ml_feature_importance.png",
    "- outputs/figures/ml_popularity_comparison.png",
    "- outputs/figures/ml_threshold_tuning.png",
    "- outputs/figures/ml_cv_metrics.png",
]

summary_path = OUTPUT_DIR / "ml_success_prediction_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print(summary_path)

C:\Users\刘子豪\Desktop\code\IMDB_analysis\outputs\ml_success_prediction_summary.md


## 14. Final Checklist

This checklist confirms that the ML extension generated the expected files and stayed within the planned scope.

In [17]:
expected_outputs = {
    "cleaned dataset reused": CLEANED_PATH,
    "generated ML notebook": PROJECT_ROOT / "notebooks" / "03_success_prediction_ml.ipynb",
    "model metrics": OUTPUT_DIR / "ml_model_metrics.csv",
    "ML summary file": OUTPUT_DIR / "ml_success_prediction_summary.md",
    "threshold tuning table": OUTPUT_DIR / "ml_threshold_tuning.csv",
    "cross-validation table": OUTPUT_DIR / "ml_cross_validation_metrics.csv",
    "model comparison figure": FIGURES_DIR / "ml_model_comparison.png",
    "confusion matrix figure": FIGURES_DIR / "ml_confusion_matrix.png",
    "feature importance figure": FIGURES_DIR / "ml_feature_importance.png",
    "popularity comparison figure": FIGURES_DIR / "ml_popularity_comparison.png",
    "threshold tuning figure": FIGURES_DIR / "ml_threshold_tuning.png",
    "cross-validation figure": FIGURES_DIR / "ml_cv_metrics.png",
    "Random Forest model": MODELS_DIR / "random_forest.joblib",
}

checklist = pd.DataFrame(
    [(name, str(path.relative_to(PROJECT_ROOT)), path.exists()) for name, path in expected_outputs.items()],
    columns=["Item", "Path", "Exists"],
)
checklist

,Item,Path,Exists
0,cleaned dataset reused,outputs\cleaned_movies.csv,True
1,generated ML notebook,notebooks\03_success_prediction_ml.ipynb,True
2,model metrics,outputs\ml_model_metrics.csv,True
3,ML summary file,outputs\ml_success_prediction_summary.md,True
4,threshold tuning table,outputs\ml_threshold_tuning.csv,True
5,cross-validation table,outputs\ml_cross_validation_metrics.csv,True
6,model comparison figure,outputs\figures\ml_model_comparison.png,True
7,confusion matrix figure,outputs\figures\ml_confusion_matrix.png,True
8,feature importance figure,outputs\figures\ml_feature_importance.png,True
9,popularity comparison figure,outputs\figures\ml_popularity_comparison.png,True
